In [1]:
import cv2
import numpy as np

In [2]:
def aruco_display(corners, ids, rejected, frame):
    image = frame.copy()
    ids = ids.flatten()
    
    for (markerCorner, markerID) in zip(corners, ids):
            
            corners = markerCorner.reshape((4, 2))
            (topLeft, topRight, bottomRight, bottomLeft) = corners
            
            topRight = (int(topRight[0]), int(topRight[1]))
            bottomRight = (int(bottomRight[0]), int(bottomRight[1]))
            bottomLeft = (int(bottomLeft[0]), int(bottomLeft[1]))
            topLeft = (int(topLeft[0]), int(topLeft[1]))

            cv2.line(image, topLeft, topRight, (0, 255, 0), 2)
            cv2.line(image, topRight, bottomRight, (0, 255, 0), 2)
            cv2.line(image, bottomRight, bottomLeft, (0, 255, 0), 2)
            cv2.line(image, bottomLeft, topLeft, (0, 255, 0), 2)
            
            cX = int((topLeft[0] + bottomRight[0]) / 2.0)
            cY = int((topLeft[1] + bottomRight[1]) / 2.0)
            cv2.circle(image, (cX, cY), 4, (0, 0, 255), -1)
            
            cv2.putText(image, str(markerID),(topLeft[0], topLeft[1] - 10), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, (0, 255, 0), 2)
            # print("[Inference] ArUco marker ID: {}".format(markerID))
            
    return image

In [3]:
Nights = cv2.VideoCapture('Videos/Nights.mp4')
Rise = cv2.VideoCapture('Videos/Rise.mp4')

videos = [Nights, Rise]

In [4]:
def video_aug(corners, ids, rejected, frame, frameCounter):
    h, w = frame.shape[:2]
    mask = np.zeros([h,w], dtype=np.uint8)
    
    warp_image = np.zeros_like(frame)
    
    for (markerCorner, markerID) in zip(corners, ids):
        try:
            if frameCounter % videos[int(markerID)].get(cv2.CAP_PROP_FRAME_COUNT) == 0:        
                videos[int(markerID)].set(cv2.CAP_PROP_POS_FRAMES, 0)
                
            _,src_image = videos[int(markerID)].read()
            src_h, src_w = src_image.shape[:2]
            # TopL, TopR, BottomR, BottomL
            src_pts = np.array([[0, 0], [src_w, 0], [src_w, src_h], [0, src_h]])

            crns = markerCorner.reshape([4, 2]).astype(int)

            H, _ = cv2.findHomography(srcPoints=src_pts, dstPoints=crns)
            augmented_image = cv2.warpPerspective(src_image, H, (w, h))
            
            cv2.fillConvexPoly(mask, crns, 255)
            frame = cv2.bitwise_and(frame, frame, mask=cv2.bitwise_not(mask))
            warp_image = cv2.bitwise_or(warp_image, augmented_image)
        except IndexError as e:
            print(f"Error: {e}, marker ID: {int(markerID)}")

    # Combine the original frame with the augmented warp_image
    final_image = cv2.bitwise_or(frame, warp_image)

    return final_image

In [5]:
# Parameters
arucoDict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
arucoParams = cv2.aruco.DetectorParameters()

In [6]:
detection = False
frameCounter = 0

cap = cv2.VideoCapture(0)

while cap.isOpened():
    
    _, img = cap.read()
        
    corners, ids, rejected = cv2.aruco.detectMarkers(img, arucoDict, parameters=arucoParams)
    
    if len(corners) > 0:
        detection = True
        
        detected_markers = aruco_display(corners, ids, rejected, img)
        cv2.imshow("Detected Markers", detected_markers)
        
        warp_video = video_aug(corners, ids, rejected, img, frameCounter)
        frameCounter += 1
        
        cv2.imshow("Warp Video", warp_video)
    else:
        detection = False
        for i in range(len(videos)):
            videos[i].set(cv2.CAP_PROP_POS_FRAMES, 0)
        
    cv2.imshow("Image", img)
    
    if cv2.waitKey(5) & 0xFF == 27:
        break
        
cap.release()
cv2.destroyAllWindows()

In [7]:
cap.release()
cv2.destroyAllWindows()